# RecordDiff - Channel-Decomposition TSTR

## 1 · Setup

In [ ]:
import os, time, numpy as np, torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)
if device == "cpu":
    print("WARNING: enable Runtime > Change runtime type > GPU (label-conditional retrain + generation).")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Configuration

In [ ]:
# paths
V2_CACHE = "..."
CKPT_C   = "..."

TASKS      = ["mortality_inhosp", "icu_mortality"]
TEST_FRAC  = 0.20
N_SYNTH    = 15000

D_Z, D_E, D_RNN, D_COMB, H_MASK, H_DEN, D_U, N_DIFF = 64, 128, 128, 128, 256, 512, 8, 100
PHASE_C_EPOCHS = (12, 6, 12)
BATCH, LR, BETA_MAX, LAMBDA_M, FREE_BITS = 256, 1e-3, 0.5, 1.0, 0.05
WINS = (0.005, 0.995)
SEED = 0
rng = np.random.default_rng(SEED)
print("tasks", TASKS, "| N_synth", N_SYNTH, "| schedule", PHASE_C_EPOCHS)

## 3 · Model core

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# utils
def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device).float() / max(half, 1))
    args = t.float()[:, None] * freqs[None]
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        emb = F.pad(emb, (0, 1))
    return emb


def causal_hist(m):
    B, T, V = m.shape
    csum = torch.cumsum(m, dim=1)
    count_excl = csum - m
    idx = torch.arange(T, device=m.device).view(1, T, 1).float().expand(B, T, V)
    obs_pos = torch.where(m > 0.5, idx, torch.full_like(m, -1.0))
    shifted = torch.cat([torch.full((B, 1, V), -1.0, device=m.device), obs_pos[:, :-1, :]], dim=1).contiguous()
    last_obs_excl = torch.cummax(shifted, dim=1).values
    dt = idx - last_obs_excl
    return torch.cat([count_excl / T, dt / T], dim=-1)


class DiffusionSchedule:
    def __init__(self, n_steps=100, beta_start=1e-4, beta_end=2e-2):
        betas = torch.linspace(beta_start, beta_end, n_steps)
        alphas = 1.0 - betas
        abar = torch.cumprod(alphas, dim=0)
        abar_prev = torch.cat([torch.ones(1), abar[:-1]])
        self.n_steps = n_steps
        self.betas = betas
        self.alphas = alphas
        self.abar = abar
        self.sqrt_abar = torch.sqrt(abar)
        self.sqrt_one_minus_abar = torch.sqrt(1.0 - abar)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
        self.posterior_var = betas * (1.0 - abar_prev) / (1.0 - abar)

    def to(self, device):
        for k, v in list(self.__dict__.items()):
            if torch.is_tensor(v):
                setattr(self, k, v.to(device))
        return self

    def q_sample(self, x0, t, noise):
        sa = self.sqrt_abar[t].view(-1, 1, 1)
        soma = self.sqrt_one_minus_abar[t].view(-1, 1, 1)
        return sa * x0 + soma * noise


class Denoiser(nn.Module):
    def __init__(self, V, d_cond, d_temb=64, hidden=256):
        super().__init__()
        self.d_temb = d_temb
        self.net = nn.Sequential(
            nn.Linear(V + d_cond + d_temb, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, V),
        )

    def forward(self, x, cond, t):
        temb = timestep_embedding(t, self.d_temb)
        temb = temb[:, None, :].expand(-1, x.shape[1], -1)
        return self.net(torch.cat([x, cond, temb], dim=-1))


# model
class RecordDiff(nn.Module):
    def __init__(self, V, d_c=8, d_z=32, d_e=128, d_rnn=128, d_comb=128,
                 hidden_mask=256, hidden_den=256, d_temb=64, n_diff=100, d_u=8):
        super().__init__()
        self.V, self.d_c, self.d_z, self.d_u = V, d_c, d_z, d_u
        self.diff = DiffusionSchedule(n_diff)

        self.emb = nn.Sequential(nn.Linear(2 * V, d_e), nn.SiLU(), nn.Linear(d_e, d_e))
        self.bigru = nn.GRU(d_e, d_rnn, batch_first=True, bidirectional=True)
        self.comb_z = nn.Linear(d_z, d_comb)
        self.comb_g = nn.Linear(2 * d_rnn, d_comb)
        self.q_mu = nn.Linear(d_comb, d_z)
        self.q_ls = nn.Linear(d_comb, d_z)
        if d_u > 0:
            self.u_mu = nn.Linear(2 * d_rnn, d_u)
            self.u_ls = nn.Linear(2 * d_rnn, d_u)
        self.p0_mu = nn.Linear(d_c, d_z)
        self.p0_ls = nn.Linear(d_c, d_z)
        self.tr_body = nn.Sequential(nn.Linear(d_z + d_c, d_comb), nn.SiLU())
        self.tr_mu = nn.Linear(d_comb, d_z)
        self.tr_ls = nn.Linear(d_comb, d_z)
        # Stage 1: measurement policy
        self.mask_head = nn.Sequential(
            nn.Linear(d_z + d_c + 2 * V + d_u, hidden_mask), nn.SiLU(),
            nn.Linear(hidden_mask, hidden_mask), nn.SiLU(),
            nn.Linear(hidden_mask, V))
        # Stage 2: value diffusion
        self.d_cond = d_z + d_c + V + 2 * V
        self.denoiser = Denoiser(V, self.d_cond, d_temb, hidden_den)
        # value normalisation
        self.register_buffer("norm_mean", torch.zeros(V))
        self.register_buffer("norm_std", torch.ones(V))

    # parameter groups for the 3-phase schedule
    def params_encoder(self):
        mods = [self.emb, self.bigru, self.comb_z, self.comb_g, self.q_mu, self.q_ls,
                self.p0_mu, self.p0_ls, self.tr_body, self.tr_mu, self.tr_ls]
        if self.d_u > 0:
            mods += [self.u_mu, self.u_ls]
        return [p for mod in mods for p in mod.parameters()]

    def params_value(self):
        return list(self.denoiser.parameters())

    def params_mask(self):
        return list(self.mask_head.parameters())

    def set_normalizer(self, mean, std):
        self.norm_mean.data = mean.to(self.norm_mean.device)
        self.norm_std.data = std.clamp(min=1e-3).to(self.norm_std.device)

    def _prior_step(self, z_prev, c, t):
        if t == 0:
            return self.p0_mu(c), self.p0_ls(c)
        body = self.tr_body(torch.cat([z_prev, c], dim=-1))
        return z_prev + self.tr_mu(body), self.tr_ls(body)

    def infer(self, y, m, c):
        B, T, V = y.shape
        e = self.emb(torch.cat([y, m], dim=-1))
        g, _ = self.bigru(e)

        if self.d_u > 0:
            gpool = g.mean(1)
            mu_u, ls_u = self.u_mu(gpool), self.u_ls(gpool)
            std_u = F.softplus(ls_u) + 1e-4
            u = mu_u + std_u * torch.randn_like(std_u)
            kl_u = (-torch.log(std_u) + 0.5 * (std_u ** 2 + mu_u ** 2) - 0.5)
        else:
            u = torch.zeros(B, 0, device=y.device)
            kl_u = torch.zeros(B, 0, device=y.device)
        z_prev = torch.zeros(B, self.d_z, device=y.device)
        Zs, KLs = [], []
        for t in range(T):
            hc = 0.5 * (torch.tanh(self.comb_z(z_prev)) + self.comb_g(g[:, t, :]))
            mu_q, ls_q = self.q_mu(hc), self.q_ls(hc)
            mu_p, ls_p = self._prior_step(z_prev, c, t)
            std_q = F.softplus(ls_q) + 1e-4
            std_p = F.softplus(ls_p) + 1e-4
            z = mu_q + std_q * torch.randn_like(std_q)
            kl = (torch.log(std_p / std_q)
                  + (std_q ** 2 + (mu_q - mu_p) ** 2) / (2 * std_p ** 2) - 0.5)
            Zs.append(z); KLs.append(kl)
            z_prev = z
        Z = torch.stack(Zs, dim=1)
        KL = torch.stack(KLs, dim=1)
        return Z, KL, u, kl_u

    def losses(self, y, m, c, free_bits=0.02):
        B, T, V = y.shape
        hist = causal_hist(m)
        Z, KL, u, kl_u = self.infer(y, m, c)
        c_seq = c[:, None, :].expand(-1, T, -1)
        u_seq = u[:, None, :].expand(-1, T, -1)

        # Stage 1: measurement policy
        mask_logits = self.mask_head(torch.cat([Z, c_seq, hist, u_seq], dim=-1))
        L_mask = F.binary_cross_entropy_with_logits(mask_logits, m)

        # Stage 2: masked conditional diffusion on observed values
        x0 = y
        tau = torch.randint(0, self.diff.n_steps, (B,), device=y.device)
        noise = torch.randn_like(x0)
        x_noisy = self.diff.q_sample(x0, tau, noise)
        cond = torch.cat([Z, c_seq, m, hist], dim=-1)
        eps_pred = self.denoiser(x_noisy, cond, tau)
        se = (eps_pred - noise) ** 2
        L_value = (se * m).sum() / m.sum().clamp(min=1.0)

        KL_fb = (torch.clamp(KL, min=free_bits).sum(-1).mean()
                 + torch.clamp(kl_u, min=free_bits).sum(-1).mean())
        return {"L_value": L_value, "L_mask": L_mask, "KL": KL_fb}

    # generation
    @torch.no_grad()
    def _ddpm_sample(self, cond):

        n = cond.shape[0]
        dev = cond.device
        x = torch.randn(n, 1, self.V, device=dev)
        cond1 = cond[:, None, :]
        for i in reversed(range(self.diff.n_steps)):
            tau = torch.full((n,), i, dtype=torch.long, device=dev)
            eps = self.denoiser(x, cond1, tau)
            mean = self.diff.sqrt_recip_alphas[i] * (
                x - self.diff.betas[i] / self.diff.sqrt_one_minus_abar[i] * eps)
            if i > 0:
                x = mean + torch.sqrt(self.diff.posterior_var[i]) * torch.randn_like(x)
            else:
                x = mean
        return x[:, 0, :]

    @torch.no_grad()
    def generate(self, n, c, T, denorm=True, return_x=False):
        dev = c.device
        V = self.V
        count_run = torch.zeros(n, V, device=dev)
        last_obs = -torch.ones(n, V, device=dev)
        z_prev = torch.zeros(n, self.d_z, device=dev)
        u = torch.randn(n, self.d_u, device=dev) if self.d_u > 0 else torch.zeros(n, 0, device=dev)
        M = torch.zeros(n, T, V, device=dev)
        Y = torch.zeros(n, T, V, device=dev)
        X = torch.zeros(n, T, V, device=dev) if return_x else None
        for t in range(T):
            mu_p, ls_p = self._prior_step(z_prev, c, t)
            z = mu_p + (F.softplus(ls_p) + 1e-4) * torch.randn_like(mu_p)
            hist = torch.cat([count_run / T, (t - last_obs) / T], dim=-1)
            m_t = torch.bernoulli(torch.sigmoid(self.mask_head(torch.cat([z, c, hist, u], dim=-1))))
            x_t = self._ddpm_sample(torch.cat([z, c, m_t, hist], dim=-1))
            if denorm:
                x_t = x_t * self.norm_std + self.norm_mean
            M[:, t, :] = m_t
            Y[:, t, :] = m_t * x_t
            if return_x:
                X[:, t, :] = x_t
            last_obs = torch.where(m_t > 0.5, torch.full_like(last_obs, float(t)), last_obs)
            count_run = count_run + m_t
            z_prev = z
        return (Y, M, X) if return_x else (Y, M)

# data helpers
def fit_normalizer(y, m):
    V = y.shape[-1]
    mean = torch.zeros(V); std = torch.ones(V)
    for v in range(V):
        vals = y[:, :, v][m[:, :, v] > 0.5]
        if vals.numel() > 10:
            mean[v] = vals.mean()
            std[v] = vals.std().clamp(min=1e-3)
    return mean, std


def normalize(y, m, mean, std):
    yn = (y - mean) / std
    return yn * m


def set_requires_grad(params, flag):
    for p in params:
        p.requires_grad_(flag)


def train_recorddiff(model, y, m, c, epochs=(8, 4, 8), batch=256, lr=1e-3,
                     beta_max=1.0, lambda_m=1.0, free_bits=0.02, val_frac=0.1,
                     clip=5.0, device="cpu", seed=0, verbose=True):

    torch.manual_seed(seed)
    model.to(device); model.diff.to(device)
    N = y.shape[0]
    perm = torch.randperm(N)
    n_val = int(N * val_frac)
    val_idx, tr_idx = perm[:n_val], perm[n_val:]
    e1, e2, e3 = epochs
    total = e1 + e2 + e3
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    mean, std = model.norm_mean.cpu(), model.norm_std.cpu()
    hist = {"phase": [], "L_value": [], "L_mask": [], "KL": [], "val_total": []}

    def run_batches(idx, train=True):
        agg = {"L_value": 0.0, "L_mask": 0.0, "KL": 0.0, "tot": 0.0, "nb": 0}
        order = idx[torch.randperm(len(idx))] if train else idx
        for s in range(0, len(order), batch):
            bi = order[s:s + batch]
            yb = normalize(y[bi], m[bi], mean, std).to(device)
            mb = m[bi].to(device)
            cb = c[bi].to(device)
            out = model.losses(yb, mb, cb, free_bits=free_bits)
            if phase == 1:
                loss = out["L_value"] + beta * out["KL"]
            elif phase == 2:
                loss = out["L_mask"]
            else:
                loss = out["L_value"] + lambda_m * out["L_mask"] + beta_max * out["KL"]
            if train:
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip); opt.step()
            for k in ("L_value", "L_mask", "KL"):
                agg[k] += float(out[k].detach())
            agg["tot"] += float(loss.detach()); agg["nb"] += 1
        for k in ("L_value", "L_mask", "KL", "tot"):
            agg[k] /= max(agg["nb"], 1)
        return agg

    for ep in range(total):
        if ep < e1:
            phase = 1; beta = beta_max * min(1.0, (ep + 1) / max(e1, 1))
            set_requires_grad(model.params_encoder(), True)
            set_requires_grad(model.params_value(), True)
            set_requires_grad(model.params_mask(), False)
        elif ep < e1 + e2:
            phase = 2; beta = beta_max
            set_requires_grad(model.params_encoder(), False)
            set_requires_grad(model.params_value(), False)
            set_requires_grad(model.params_mask(), True)
        else:
            phase = 3; beta = beta_max
            set_requires_grad(model.params_encoder(), True)
            set_requires_grad(model.params_value(), True)
            set_requires_grad(model.params_mask(), True)

        model.train(); tr = run_batches(tr_idx, train=True)
        model.eval()
        with torch.no_grad():
            va = run_batches(val_idx, train=False) if n_val > 0 else tr
        hist["phase"].append(phase)
        hist["L_value"].append(tr["L_value"]); hist["L_mask"].append(tr["L_mask"])
        hist["KL"].append(tr["KL"]); hist["val_total"].append(va["tot"])
        if verbose:
            print(f"ep {ep:3d} | phase {phase} | "
                  f"L_value {tr['L_value']:.4f}  L_mask {tr['L_mask']:.4f}  KL {tr['KL']:.4f} "
                  f"| val_total {va['tot']:.4f}")
    return hist


# synthetic (for smoke test)
def make_synthetic(n=512, T=12, V=6, seed=0, device="cpu"):
    """Toy MNAR data: latent severity drives both values and (informatively) the mask."""
    g = torch.Generator().manual_seed(seed)
    sev = torch.rand(n, 1, 1, generator=g)
    base = torch.randn(n, 1, V, generator=g)
    drift = torch.linspace(0, 1, T).view(1, T, 1) * (sev - 0.5) * 4.0
    x = base + drift + 0.3 * torch.randn(n, T, V, generator=g)
    logit = -0.5 + 2.0 * sev + 0.6 * (x > 1.0).float() + 0.4 * torch.randn(n, T, V, generator=g)
    m = torch.bernoulli(torch.sigmoid(logit), generator=g)
    y = x * m
    c = torch.cat([sev.view(n, 1), torch.randn(n, 7, generator=g)], dim=-1)
    return y.to(device), m.to(device), c.to(device)


## 4 · TSTR helpers

In [ ]:

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score


# featurizers
def mask_features(m):

    N, T, V = m.shape
    count = m.sum(1).astype(np.float32)
    rate = count / T
    obs = m > 0.5
    first = np.where(obs.any(1), np.argmax(obs, 1), T).astype(np.float32) / T
    rev = obs[:, ::-1, :]
    last_idx = np.where(obs.any(1), (T - 1 - np.argmax(rev, 1)), -1.0).astype(np.float32)
    last = (last_idx + 1.0) / T
    return np.concatenate([count, rate, first, last], axis=1).astype(np.float32)


def value_features(y, m):

    N, T, V = y.shape
    obs = m > 0.5
    cnt = np.maximum(m.sum(1), 1)
    mean = (y.sum(1) / cnt).astype(np.float32)
    var = np.maximum((y * y).sum(1) / cnt - mean ** 2, 0.0)
    std = np.sqrt(var).astype(np.float32)
    any_obs = obs.any(1)
    vmin = np.where(any_obs, np.where(obs, y, np.inf).min(1), 0.0).astype(np.float32)
    vmax = np.where(any_obs, np.where(obs, y, -np.inf).max(1), 0.0).astype(np.float32)
    rev = obs[:, ::-1, :]
    last_t = np.where(any_obs, T - 1 - np.argmax(rev, 1), 0)
    last = np.take_along_axis(y, last_t[:, None, :], axis=1)[:, 0, :]
    last = np.where(any_obs, last, 0.0).astype(np.float32)
    feats = np.concatenate([mean, std, vmin, vmax, last], axis=1)
    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)


def channel_features(m, y, channel):

    parts = []
    if channel in ("mask", "joint"):
        parts.append(mask_features(m))
    if channel in ("value", "joint"):
        parts.append(value_features(y, m))
    return np.nan_to_num(np.concatenate(parts, axis=1))


# TSTR
def _clf(seed):
    return HistGradientBoostingClassifier(max_depth=4, max_iter=200, learning_rate=0.1,
                                          random_state=seed)


def transfer_auroc(m_tr, y_tr, lab_tr, m_te, y_te, lab_te, channel, seed=0, n_boot=300):

    Xtr = channel_features(m_tr, y_tr, channel)
    Xte = channel_features(m_te, y_te, channel)
    sc = StandardScaler().fit(Xtr)
    clf = _clf(seed).fit(sc.transform(Xtr), lab_tr)
    p = clf.predict_proba(sc.transform(Xte))[:, 1]
    auc = roc_auc_score(lab_te, p)
    rng = np.random.default_rng(seed); n = len(lab_te); boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(lab_te[idx])) == 2:
            boots.append(roc_auc_score(lab_te[idx], p[idx]))
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (auc, auc))
    return float(auc), float(lo), float(hi)


## 5 · Smoke test

In [ ]:
ys, ms, cs = make_synthetic(n=400, T=10, V=6, seed=0)
lab_toy = (np.random.default_rng(0).random(400) < 0.4).astype(int)
c_aug_toy = np.concatenate([cs.numpy(), lab_toy[:, None].astype(np.float32)], axis=1)  # d_c = 9
sm = RecordDiff(V=6, d_c=9, d_z=8, d_e=32, d_rnn=32, d_comb=32,
                hidden_mask=64, hidden_den=64, d_temb=32, n_diff=15, d_u=4).to(device)
sm.diff.to(device); sm.set_normalizer(*fit_normalizer(ys, ms))
train_recorddiff(sm, ys, ms, torch.from_numpy(c_aug_toy), epochs=(2, 1, 2), batch=128,
                 beta_max=0.5, free_bits=0.05, device=device, verbose=False)
sm.eval(); sm.diff.to(device)
Yg, Mg, Xg = sm.generate(400, torch.from_numpy(c_aug_toy).to(device), T=10, return_x=True)
Mg, Yg = Mg.cpu().numpy(), Yg.cpu().numpy()
a, lo, hi = transfer_auroc(Mg, Yg, lab_toy, Mg, Yg, lab_toy, "joint")
assert 0.0 <= a <= 1.0 and Mg.shape == (400, 10, 6), "smoke failed"
print(f"SMOKE OK — label-conditional train+generate+transfer run cleanly (sanity AUROC {a:.3f}).")

## 6 · Load cohort, build label-conditional `c`, split real train/test

In [ ]:
d = np.load(V2_CACHE, allow_pickle=True)
m_all, y_all = d["m"].astype(np.float32), d["y"].astype(np.float32)
VAR_NAMES = list(d["var_names"]); VAR_CLASS = [str(x) for x in d["var_class"]]
c_cov = d["c"].astype(np.float32) if "c" in d.files else np.zeros((len(m_all), 0), np.float32)
N, T, V = m_all.shape
labels = {t: d[f"lab_{t}"].astype(int) for t in TASKS}
lab_mat = np.stack([labels[t] for t in TASKS], axis=1).astype(np.float32)
c_aug = np.concatenate([c_cov, lab_mat], axis=1)          # covariates ++ labels
D_C_AUG = c_aug.shape[1]

tr_idx, te_idx = train_test_split(np.arange(N), test_size=TEST_FRAC,
                                  stratify=labels[TASKS[0]], random_state=SEED)

m_all_w, y_all_w = m_all.copy(), y_all.copy()
for v in range(V):
    obs = y_all[tr_idx][:, :, v][m_all[tr_idx][:, :, v] > 0.5]
    if obs.size > 200:
        lo, hi = np.quantile(obs, WINS)
        y_all_w[:, :, v] = np.clip(y_all_w[:, :, v], lo, hi) * m_all[:, :, v]

m_tr, y_tr, c_tr = m_all_w[tr_idx], y_all_w[tr_idx], c_aug[tr_idx]
m_te, y_te = m_all_w[te_idx], y_all_w[te_idx]
print(f"cohort N={N} | train {len(tr_idx)} | test {len(te_idx)} | d_c_aug={D_C_AUG}")
for t in TASKS:
    print(f"  {t:18s} prevalence  train {labels[t][tr_idx].mean():.3f} | test {labels[t][te_idx].mean():.3f}")

## 7 · Retrain RecordDiff label-conditionally

In [ ]:
model = RecordDiff(V=V, d_c=D_C_AUG, d_z=D_Z, d_e=D_E, d_rnn=D_RNN, d_comb=D_COMB,
                   hidden_mask=H_MASK, hidden_den=H_DEN, d_temb=64, n_diff=N_DIFF, d_u=D_U).to(device)
model.diff.to(device)
mean, std = fit_normalizer(torch.from_numpy(y_tr), torch.from_numpy(m_tr))
model.set_normalizer(mean, std)
t0 = time.time()
hist = train_recorddiff(model, torch.from_numpy(y_tr), torch.from_numpy(m_tr), torch.from_numpy(c_tr),
                        epochs=PHASE_C_EPOCHS, batch=BATCH, lr=LR, beta_max=BETA_MAX,
                        lambda_m=LAMBDA_M, free_bits=FREE_BITS, device=device, seed=SEED, verbose=True)
print(f"\ntrained in {(time.time()-t0)/60:.1f} min")
torch.save({"state_dict": model.state_dict(),
            "config": dict(V=V, d_c=D_C_AUG, d_z=D_Z, d_e=D_E, d_rnn=D_RNN, d_comb=D_COMB,
                           hidden_mask=H_MASK, hidden_den=H_DEN, d_temb=64, n_diff=N_DIFF, d_u=D_U),
            "tasks": TASKS, "norm_mean": mean, "norm_std": std}, CKPT_C)
print("label-conditional checkpoint saved ->", CKPT_C)

## 8 · Generate labeled synthetic + the mask-agnostic baseline

In [ ]:
sidx = rng.choice(tr_idx, size=min(N_SYNTH, len(tr_idx)), replace=False)
c_gen = torch.from_numpy(c_aug[sidx]).to(device)
model.eval(); model.diff.to(device)
t0 = time.time()
with torch.no_grad():
    Yg, Mg, Xg = model.generate(len(sidx), c_gen, T, return_x=True)
Y_rd, M_rd, X_rd = Yg.cpu().numpy(), Mg.cpu().numpy(), Xg.cpu().numpy()
print(f"generated {len(sidx)} label-conditional records in {time.time()-t0:.0f}s")

DISCRETE = {"GCS_eye": (1, 4), "GCS_verbal": (1, 5), "GCS_motor": (1, 6)}
for v in range(V):
    obs = y_all[tr_idx][:, :, v][m_all[tr_idx][:, :, v] > 0.5]
    if obs.size > 200:
        lo, hi = np.quantile(obs, WINS); X_rd[:, :, v] = np.clip(X_rd[:, :, v], lo, hi)
for nm, (lo, hi) in DISCRETE.items():
    if nm in VAR_NAMES:
        vi = VAR_NAMES.index(nm); X_rd[:, :, vi] = np.clip(np.round(X_rd[:, :, vi]), lo, hi)
Y_rd = (M_rd * X_rd).astype(np.float32)

marg = m_tr.mean((0, 1))
M_ma = (rng.random((len(sidx), T, V)) < marg[None, None, :]).astype(np.float32)
Y_ma = (M_ma * X_rd).astype(np.float32)
print(f"cohorts ready | RecordDiff mask density {M_rd.mean():.3f} | mask-agnostic {M_ma.mean():.3f}")

## 9 · Channel-decomposition TSTR

In [ ]:
results = {}
for task in TASKS:
    lr_ = labels[task][tr_idx]; lte = labels[task][te_idx]; lsyn = labels[task][sidx]
    results[task] = {}
    print(f"\n=== {task} ===")
    print(f"{'channel':7s} | {'TRTR (real->real)':>20s} {'RecordDiff TSTR':>20s} {'mask-agnostic TSTR':>20s}")
    print("-" * 74)
    for ch in ["mask", "value", "joint"]:
        trtr = transfer_auroc(m_tr, y_tr, lr_, m_te, y_te, lte, ch, seed=SEED)
        rd = transfer_auroc(M_rd, Y_rd, lsyn, m_te, y_te, lte, ch, seed=SEED)
        ma = transfer_auroc(M_ma, Y_ma, lsyn, m_te, y_te, lte, ch, seed=SEED)
        results[task][ch] = {"trtr": trtr, "rd": rd, "ma": ma}
        f = lambda t: f"{t[0]:.3f} [{t[1]:.2f},{t[2]:.2f}]"
        print(f"{ch:7s} | {f(trtr):>20s} {f(rd):>20s} {f(ma):>20s}")

In [ ]:
import torch.nn as nn

BASE_EPOCHS, BASE_HID = 12, 192

class ValueDiffBaseline(nn.Module):
    def __init__(self, V, d_c, d_temb=64, hidden=192, n_diff=100):
        super().__init__()
        self.V, self.d_temb = V, d_temb
        self.diff = DiffusionSchedule(n_diff)
        self.inp = nn.Linear(V + d_c + d_temb, hidden)
        self.gru = nn.GRU(hidden, hidden, batch_first=True, bidirectional=True)
        self.out = nn.Sequential(nn.SiLU(), nn.Linear(2 * hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, V))
        self.register_buffer("nm", torch.zeros(V)); self.register_buffer("ns", torch.ones(V))
    def set_norm(self, m, s):
        self.nm.data = m.to(self.nm.device); self.ns.data = s.clamp(min=1e-3).to(self.ns.device)
    def _eps(self, x, c, t):
        temb = timestep_embedding(t, self.d_temb)[:, None, :].expand(-1, x.shape[1], -1)
        cc = c[:, None, :].expand(-1, x.shape[1], -1)
        o, _ = self.gru(torch.relu(self.inp(torch.cat([x, cc, temb], dim=-1))))
        return self.out(o)
    def loss(self, y, m, c):
        B = y.shape[0]
        tau = torch.randint(0, self.diff.n_steps, (B,), device=y.device)
        noise = torch.randn_like(y)
        ep = self._eps(self.diff.q_sample(y, tau, noise), c, tau)
        return ((ep - noise) ** 2 * m).sum() / m.sum().clamp(min=1.0)
    @torch.no_grad()
    def generate(self, n, c, T):
        dev = c.device
        x = torch.randn(n, T, self.V, device=dev)
        for i in reversed(range(self.diff.n_steps)):
            tau = torch.full((n,), i, dtype=torch.long, device=dev)
            ep = self._eps(x, c, tau)
            mean = self.diff.sqrt_recip_alphas[i] * (
                x - self.diff.betas[i] / self.diff.sqrt_one_minus_abar[i] * ep)
            x = mean + (torch.sqrt(self.diff.posterior_var[i]) * torch.randn_like(x) if i > 0 else 0.0)
        return x * self.ns + self.nm

def train_baseline(model, y, m, c, epochs=12, batch=256, lr=1e-3):
    model.to(device); model.diff.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    mean, std = model.nm.cpu(), model.ns.cpu(); Nn = y.shape[0]
    for ep in range(epochs):
        model.train(); perm = torch.randperm(Nn); tot = nb = 0
        for s in range(0, Nn, batch):
            bi = perm[s:s + batch]
            yb = normalize(y[bi], m[bi], mean, std).to(device); mb = m[bi].to(device); cb = c[bi].to(device)
            loss = model.loss(yb, mb, cb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
            tot += float(loss); nb += 1
        if (ep + 1) % 4 == 0 or ep == epochs - 1:
            print(f"  value-diffusion baseline | epoch {ep:2d} | loss {tot/nb:.4f}")

print("training value-only diffusion baseline ...")
base = ValueDiffBaseline(V, D_C_AUG, hidden=BASE_HID, n_diff=N_DIFF).to(device)
base.set_norm(*fit_normalizer(torch.from_numpy(y_tr), torch.from_numpy(m_tr)))
t0 = time.time()
train_baseline(base, torch.from_numpy(y_tr), torch.from_numpy(m_tr), torch.from_numpy(c_tr), epochs=BASE_EPOCHS)
print(f"  trained in {(time.time()-t0)/60:.1f} min")

base.eval(); base.diff.to(device)
with torch.no_grad():
    Xb = base.generate(len(sidx), torch.from_numpy(c_aug[sidx]).to(device), T).cpu().numpy()
for v in range(V):
    obs = y_all[tr_idx][:, :, v][m_all[tr_idx][:, :, v] > 0.5]
    if obs.size > 200:
        lo, hi = np.quantile(obs, WINS); Xb[:, :, v] = np.clip(Xb[:, :, v], lo, hi)
for nm_, (lo, hi) in {"GCS_eye": (1, 4), "GCS_verbal": (1, 5), "GCS_motor": (1, 6)}.items():
    if nm_ in VAR_NAMES:
        vi = VAR_NAMES.index(nm_); Xb[:, :, vi] = np.clip(np.round(Xb[:, :, vi]), lo, hi)
M_base = (rng.random((len(sidx), T, V)) < marg[None, None, :]).astype(np.float32)
Y_base = (M_base * Xb).astype(np.float32)

for task in TASKS:
    lte = labels[task][te_idx]; lsyn = labels[task][sidx]
    print(f"\n=== {task} — with value-only diffusion baseline ===")
    print(f"{'channel':7s} | {'TRTR':>16s} {'RecordDiff':>16s} {'value-diff base':>16s} {'mask-agnostic':>16s}")
    print("-" * 78)
    for ch in ["mask", "value", "joint"]:
        results[task][ch]["base"] = transfer_auroc(M_base, Y_base, lsyn, m_te, y_te, lte, ch, seed=SEED)
        f = lambda t: f"{t[0]:.3f}"
        r = results[task][ch]
        print(f"{ch:7s} | {f(r['trtr']):>16s} {f(r['rd']):>16s} {f(r['base']):>16s} {f(r['ma']):>16s}")

fig, axes = plt.subplots(1, len(TASKS), figsize=(7.5 * len(TASKS), 5), squeeze=False)
chans = ["mask", "value", "joint"]; xpos = np.arange(len(chans)); w = 0.2
series = [("trtr", "TRTR (ceiling)", "gray"), ("rd", "RecordDiff", "teal"),
          ("base", "value-only diffusion", "orange"), ("ma", "mask-agnostic", "red")]
for ax, task in zip(axes[0], TASKS):
    for j, (key, lab, col) in enumerate(series):
        vals = [results[task][c][key][0] for c in chans]
        err = [[results[task][c][key][0] - results[task][c][key][1] for c in chans],
               [results[task][c][key][2] - results[task][c][key][0] for c in chans]]
        ax.bar(xpos + (j - 1.5) * w, vals, w, yerr=err, capsize=2, label=lab, color=col)
    ax.axhline(0.5, ls="--", c="k", lw=1, alpha=0.6)
    ax.set_xticks(xpos); ax.set_xticklabels(chans); ax.set_ylim(0.45, 1.0)
    ax.set_ylabel("AUROC (test on real)"); ax.set_title(task); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print("\nExpectation: the value-only diffusion baseline collapses on the MASK channel (like the")
print("ablation), confirming the result holds against a genuine, independently-trained model;")
print("its value channel shows how RecordDiff's values compare to a dedicated value generator.")